# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Big Data** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Examples on Structured Streaming (files)** </center>
---
**Profesor**: Pablo Camarillo Ramirez

# Create SparkSession

In [3]:
from SparkUtils import SparkUtils

import pyspark.sql.functions as F
from pathlib import Path
import shutil

In [2]:
MASTER_URL = "spark://spark-master:7077"
APP_NAME = "Example: Structured Streaming with Files"

spark = SparkUtils(MASTER_URL, APP_NAME)._spark

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/07 00:45:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Create a data stream from a local socket

### Connect Spark to the socket

In [6]:
!ls /opt/spark/work-dir/data/streaming/logs/

In [7]:
logs_schema = SparkUtils.generate_schema([("raw_line", "string")])

input_path = "/opt/spark/work-dir/data/streaming/logs/"

# Create the stream
logs_df = (spark.readStream
    .format("text")
    .option("maxFilesPerTrigger", 1) # Let's process one file at a time
    .schema(logs_schema)
    .load(input_path))

# Transform original dataframe
parsed_df = (
    logs_df
    .withColumn("parts",     F.split(F.col("raw_line"), r" \| "))
    .withColumn("timestamp", F.to_timestamp(F.col("parts")[0], "yyyy-MM-dd HH:mm:ss"))
    .withColumn("level",     F.trim(F.col("parts")[1]))
    .withColumn("message",   F.trim(F.col("parts")[2]))
    .withColumn("server",    F.trim(F.col("parts")[3]))
    .drop("parts", "raw_line")               # keep only the clean columns
    .filter(F.col("timestamp").isNotNull())  # skip malformed lines
)

# Let's create a summary
summary_df = (
    parsed_df
    .groupBy("server", "level")
    .count()
    .orderBy("server", "level")
)

# Clean checkpoint
checkpoint_path = "/opt/spark/work-dir/checkpoints/logs_checkpoint"
dir_path = Path(checkpoint_path)
if dir_path.exists() and dir_path.is_dir():
    shutil.rmtree(dir_path)

# Write stream in the destination
query_events = (
    parsed_df.writeStream
    .outputMode("append")        # append: show new rows only
    .format("console")
    .option("truncate", False)   # don't cut off long messages
    .option("numRows", 20)
    .option("checkpointLocation", checkpoint_path)
    .queryName("parsed_logs")
    .start()
)

query_summary = (
    summary_df.writeStream
    .outputMode("complete")
    .format("console")
    .option("truncate", False)
    .queryName("summary_logs")
    .start()
)

print("   Press Ctrl+C to stop.\n")

spark.streams.awaitAnyTermination()

-------------------------------------------
Batch: 6
-------------------------------------------
+-------------------+-----+-----------------------------+-------------+
|timestamp          |level|message                      |server       |
+-------------------+-----+-----------------------------+-------------+
|2026-04-07 01:10:48|WARN |Disk usage 85%               |server-node-5|
|2026-04-07 01:10:57|INFO |Configuration reloaded       |server-node-5|
|2026-04-07 01:11:03|ERROR|Authentication failed        |server-node-2|
|2026-04-07 01:11:07|INFO |Configuration reloaded       |server-node-3|
|2026-04-07 01:11:13|INFO |Configuration reloaded       |server-node-3|
|2026-04-07 01:11:16|INFO |Configuration reloaded       |server-node-2|
|2026-04-07 01:11:19|ERROR|Disk full                    |server-node-1|
|2026-04-07 01:11:21|WARN |Response time above threshold|server-node-2|
|2026-04-07 01:11:24|INFO |Health check passed          |server-node-4|
|2026-04-07 01:11:27|ERROR|500 Internal

-------------------------------------------
Batch: 7
-------------------------------------------
+-------------------+-----+------------------------------------+-------------+
|timestamp          |level|message                             |server       |
+-------------------+-----+------------------------------------+-------------+
|2026-04-07 01:10:50|WARN |Certificate expires in 7 days       |server-node-2|
|2026-04-07 01:11:00|WARN |Certificate expires in 7 days       |server-node-4|
|2026-04-07 01:11:08|WARN |CPU spike detected                  |server-node-5|
|2026-04-07 01:11:13|ERROR|500 Internal Server Error           |server-node-5|
|2026-04-07 01:11:21|INFO |Backup completed successfully       |server-node-1|
|2026-04-07 01:11:31|INFO |Configuration reloaded              |server-node-1|
|2026-04-07 01:11:32|ERROR|Unhandled exception in worker thread|server-node-1|
|2026-04-07 01:11:41|WARN |CPU spike detected                  |server-node-1|
|2026-04-07 01:11:48|INFO |User lo

-------------------------------------------
Batch: 8
-------------------------------------------
+-------------------+-----+------------------------------------+-------------+
|timestamp          |level|message                             |server       |
+-------------------+-----+------------------------------------+-------------+
|2026-04-07 01:10:52|WARN |Response time above threshold       |server-node-2|
|2026-04-07 01:10:59|INFO |Backup completed successfully       |server-node-2|
|2026-04-07 01:11:03|WARN |Disk usage 85%                      |server-node-3|
|2026-04-07 01:11:11|WARN |Retry attempt 3 of 5                |server-node-5|
|2026-04-07 01:11:13|WARN |Certificate expires in 7 days       |server-node-1|
|2026-04-07 01:11:14|WARN |Certificate expires in 7 days       |server-node-3|
|2026-04-07 01:11:20|WARN |Memory usage 90%                    |server-node-2|
|2026-04-07 01:11:22|ERROR|Unhandled exception in worker thread|server-node-5|
|2026-04-07 01:11:26|ERROR|404 Not

-------------------------------------------
Batch: 10
-------------------------------------------
+-------------------+-----+-----------------------------+-------------+
|timestamp          |level|message                      |server       |
+-------------------+-----+-----------------------------+-------------+
|2026-04-07 01:10:56|ERROR|Disk full                    |server-node-4|
|2026-04-07 01:11:04|ERROR|Authentication failed        |server-node-4|
|2026-04-07 01:11:12|INFO |Configuration reloaded       |server-node-2|
|2026-04-07 01:11:17|WARN |Memory usage 90%             |server-node-2|
|2026-04-07 01:11:25|WARN |CPU spike detected           |server-node-5|
|2026-04-07 01:11:26|WARN |CPU spike detected           |server-node-3|
|2026-04-07 01:11:34|WARN |Disk usage 85%               |server-node-3|
|2026-04-07 01:11:44|INFO |User login successful        |server-node-3|
|2026-04-07 01:11:46|INFO |Backup completed successfully|server-node-4|
|2026-04-07 01:11:48|ERROR|500 Interna

-------------------------------------------
Batch: 7
-------------------------------------------
+-------------+-----+-----+
|server       |level|count|
+-------------+-----+-----+
|server-node-1|ERROR|8    |
|server-node-1|INFO |7    |
|server-node-1|WARN |5    |
|server-node-2|ERROR|3    |
|server-node-2|INFO |5    |
|server-node-2|WARN |5    |
|server-node-3|ERROR|6    |
|server-node-3|INFO |6    |
|server-node-3|WARN |4    |
|server-node-4|ERROR|3    |
|server-node-4|INFO |11   |
|server-node-4|WARN |5    |
|server-node-5|ERROR|4    |
|server-node-5|INFO |3    |
|server-node-5|WARN |5    |
+-------------+-----+-----+



-------------------------------------------
Batch: 8
-------------------------------------------
+-------------+-----+-----+
|server       |level|count|
+-------------+-----+-----+
|server-node-1|ERROR|8    |
|server-node-1|INFO |7    |
|server-node-1|WARN |6    |
|server-node-2|ERROR|3    |
|server-node-2|INFO |7    |
|server-node-2|WARN |7    |
|server-node-3|ERROR|7    |
|server-node-3|INFO |6    |
|server-node-3|WARN |6    |
|server-node-4|ERROR|3    |
|server-node-4|INFO |11   |
|server-node-4|WARN |5    |
|server-node-5|ERROR|5    |
|server-node-5|INFO |3    |
|server-node-5|WARN |6    |
+-------------+-----+-----+



-------------------------------------------
Batch: 9
-------------------------------------------
+-------------+-----+-----+
|server       |level|count|
+-------------+-----+-----+
|server-node-1|ERROR|8    |
|server-node-1|INFO |8    |
|server-node-1|WARN |6    |
|server-node-2|ERROR|5    |
|server-node-2|INFO |7    |
|server-node-2|WARN |10   |
|server-node-3|ERROR|8    |
|server-node-3|INFO |6    |
|server-node-3|WARN |6    |
|server-node-4|ERROR|3    |
|server-node-4|INFO |11   |
|server-node-4|WARN |5    |
|server-node-5|ERROR|5    |
|server-node-5|INFO |4    |
|server-node-5|WARN |8    |
+-------------+-----+-----+

-------------------------------------------
Batch: 10
-------------------------------------------
+-------------+-----+-----+
|server       |level|count|
+-------------+-----+-----+
|server-node-1|ERROR|8    |
|server-node-1|INFO |8    |
|server-node-1|WARN |6    |
|server-node-2|ERROR|6    |
|server-node-2|INFO |8    |
|server-node-2|WARN |11   |
|server-node-3|ERROR

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/opt/spark/python/lib/py4j-0.10.9.9-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/opt/spark/python/lib/py4j-0.10.9.9-src.zip/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/usr/lib/python3.10/socket.py", line 705, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

In [8]:
spark.stop()